In [0]:
from pyspark.sql import functions as F

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table= f"{catalog_name}.{bronze_schema}.results"
silver_table= f"{catalog_name}.{silver_schema}.results"

In [0]:
results_df = spark.read.table(bronze_table)

In [0]:
results_drop_df= results_df.drop("url")

In [0]:
results_rename_df= (results_drop_df
                    .withColumnsRenamed({
                        "constructorId": "constructor_id", 
                        "driverId": "driver_id", 
                        "raceName": "race_name", 
                        "positionText": "final_position_text",
                        "date": "race_date",
                        "grid": "grid_position",
                        "laps": "completed_laps",
                        "number": "car_number",
                        "position": "final_position"
                        })
)

In [0]:
results_valid_df= results_rename_df.filter(F.col("season").isNotNull() &
                                           F.col("round").isNotNull() & 
                                           F.col("constructor_id").isNotNull() &
                                           F.col("driver_id").isNotNull()
                                           )

In [0]:
results_dup_df = results_valid_df.dropDuplicates(["season", "round", "constructor_id", "driver_id"])

In [0]:
results_final_df= (
    results_dup_df
    .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))